# Assignment 10

### Imports

In [21]:
import os
import torch
import torch.nn as nn
import numpy as np
import json
import sys
from sklearn.preprocessing import StandardScaler

sys.path.append("../../MainProject/Assignment9")
from assignment9_functions import *

device = torch.device("cpu")
random_seed = 42
np.random.seed(random_seed)
torch.manual_seed(random_seed)

### Load kinect data

In [22]:
kinect_data = "../../MainProject/Assignment9/data/kinect_good_preprocessed"
train_files, test_files = split_csvfiles(kinect_data, random_seed, 0.9, 0)

train_data = load(train_files, kinect_data)
test_data = load(test_files, kinect_data)
x_train, y_train = input_target_split(train_data)
x_test, y_test = input_target_split(test_data)

# Data is not normalized
print(np.mean(x_train, axis=0))

head_x             -0.005158
head_y              0.499413
left_shoulder_x    -0.152877
left_shoulder_y     0.303871
left_elbow_x       -0.267661
left_elbow_y        0.492131
right_shoulder_x    0.153000
right_shoulder_y    0.302793
right_elbow_x       0.273399
right_elbow_y       0.491542
left_hand_x        -0.287320
left_hand_y         0.711735
right_hand_x        0.297538
right_hand_y        0.709301
left_hip_x         -0.071477
left_hip_y         -0.194162
right_hip_x         0.078206
right_hip_y        -0.193390
left_knee_x        -0.142173
left_knee_y        -0.444181
right_knee_x        0.146615
right_knee_y       -0.452019
left_foot_x        -0.142451
left_foot_y        -0.817323
right_foot_x        0.147881
right_foot_y       -0.826109
dtype: float64


### Baseline model from Assignment 9

In [23]:
X_train = torch.tensor(x_train.values, dtype=torch.float32).to(device)
Y_train = torch.tensor(y_train.values, dtype=torch.float32).to(device)
X_test = torch.tensor(x_test.values, dtype=torch.float32).to(device)
Y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)

# Load Baseline model
model_root = "../z_pred_models"
metadata_path = os.path.join(model_root, "metadata", "champion_info.json")
model_path = os.path.join(model_root, "champion", "champion_model.pt")

# Load baseline model
with open(metadata_path, "r") as f:
    champion_info = json.load(f)

best_config = champion_info["hyperparameters"]

best_model = build_model(best_config, device)


# Train model
result = train_one_model(
    best_model,
    best_config,
    X_train,
    Y_train,
    X_train,
    Y_train,
)
best_model.load_state_dict(result["best_state"])


loss_fn = nn.MSELoss()

# ---- TRAIN EVALUATION ----
train_loss, train_metrics, _ = evaluate_model(
    best_model,
    X_train,
    Y_train,
    loss_fn
)

# ---- TEST EVALUATION ----
test_loss, test_metrics, test_predictions = evaluate_model(
    best_model,
    X_test,
    Y_test,
    loss_fn
)

### Normalize input data

In [24]:
# Normalize input based on training data
scaler = StandardScaler()
scaler.set_output(transform="pandas")
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

### Baseline trained on normalized data

In [25]:
X_train = torch.tensor(x_train_scaled.values, dtype=torch.float32).to(device)
Y_train = torch.tensor(y_train.values, dtype=torch.float32).to(device)
X_test = torch.tensor(x_test_scaled.values, dtype=torch.float32).to(device)
Y_test = torch.tensor(y_test.values, dtype=torch.float32).to(device)


best_model_scaled = build_model(best_config, device)


# Train model
result_scaled = train_one_model(
    best_model,
    best_config,
    X_train,
    Y_train,
    X_train,
    Y_train,
)
best_model_scaled.load_state_dict(result_scaled["best_state"])


loss_fn = nn.MSELoss()

# ---- TRAIN EVALUATION ----
train_loss_scaled, train_metrics_scaled, _ = evaluate_model(
    best_model,
    X_train,
    Y_train,
    loss_fn
)

# ---- TEST EVALUATION ----
test_loss_scaled, test_metrics_scaled, test_predictions_scaled = evaluate_model(
    best_model,
    X_test,
    Y_test,
    loss_fn
)


### Compare if preprocessing alters the baseline model performance

In [26]:
print("Train metrics")
print(f"Baseline no preprocessing")
print(f"Train Loss (MSE):  {train_metrics['mse']:.6f}")
print(f"Train MAE:  {train_metrics['mae']:.6f}")
print(f"Train R2:   {train_metrics['r2']:.6f}")
print(f"Train Bias: {train_metrics['bias']:.6f}")

print(f"\nBaseline with preprocessing")
print(f"Train Loss (MSE):  {train_metrics_scaled['mse']:.6f}")
print(f"Train MAE:  {train_metrics_scaled['mae']:.6f}")
print(f"Train R2:   {train_metrics_scaled['r2']:.6f}")
print(f"Train Bias: {train_metrics_scaled['bias']:.6f}")

print("\nTest metrics")
print("Baseline no preprocessing")
print(f"Test Loss (MSE):  {test_metrics['mse']:.6f}")
print(f"Test MAE:  {test_metrics['mae']:.6f}")
print(f"Test R2:   {test_metrics['r2']:.6f}")
print(f"Test Bias: {test_metrics['bias']:.6f}")

print("\nBaseline with preprocessing")
print(f"Test Loss (MSE):  {test_metrics_scaled['mse']:.6f}")
print(f"Test MAE:  {test_metrics_scaled['mae']:.6f}")
print(f"Test R2:   {test_metrics_scaled['r2']:.6f}")
print(f"Test Bias: {test_metrics_scaled['bias']:.6f}")

Train metrics
Baseline no preprocessing
Train Loss (MSE):  0.002249
Train MAE:  0.035021
Train R2:   0.750941
Train Bias: -0.001026

Baseline with preprocessing
Train Loss (MSE):  0.002554
Train MAE:  0.036824
Train R2:   0.684499
Train Bias: -0.000938

Test metrics
Baseline no preprocessing
Test Loss (MSE):  0.004453
Test MAE:  0.050552
Test R2:   0.555981
Test Bias: 0.025352

Baseline with preprocessing
Test Loss (MSE):  0.007231
Test MAE:  0.062149
Test R2:   0.303433
Test Bias: 0.026387
